In [1]:
import pandas as pd, datetime, glob, os
pd.options.display.float_format = '{:,.2f}'.format

In [2]:
csv_files = sorted(glob.glob(os.path.join("tefasbefas2files", "*.csv")))
dfs = {os.path.basename(f).split('.')[0]: pd.read_csv(f).fillna(0) for f in csv_files}

dag_files = ['tefas_dag_1','tefas_dag_2','tefas_dag_3']

t_dag_df = pd.concat([dfs[f] for f in dag_files], ignore_index=True)
t_fon_df = dfs['tefas_fon']
b_dag_df = dfs['befas_dag']
b_fon_df = dfs['befas_fon']
dfs. clear()

#dates = [pd.to_datetime(key, format='%d.%m.%Y') for key in b_dag_df['Tarih'].unique().tolist()]
dates = [i for i in b_dag_df['Tarih'].unique().tolist()]
t_dag_df.columns = [col[:-4] if col.endswith(' (%)') else col for col in t_dag_df.columns]
b_dag_df.columns = [col[:-4] if col.endswith(' (%)') else col for col in b_dag_df.columns]
assets = list(b_dag_df.iloc[0].index)[3:]


KeyError: 'tefas_dag_1'

In [3]:
columns_to_float = ['Fiyat','Tedavüldeki Pay Sayısı']
t_df = t_fon_df.merge(t_dag_df, on=['Tarih','Fon Kodu'], how='left', suffixes=('', '_drop'))
t_dag_df =  pd.DataFrame()
f_don_df = pd.DataFrame()
t_df = t_df.drop(columns=[col for col in t_df.columns if col.endswith('_drop')])
t_df[columns_to_float+assets] = (t_df[columns_to_float+assets]
                                    .replace(r"\.", "", regex=True)
                                    .replace(",", ".", regex=True)
                                    .astype(float))
b_df = b_fon_df.merge(b_dag_df, on=['Tarih','Fon Kodu'], how='left', suffixes=('', '_drop'))
b_dag_df =  pd.DataFrame()
b_don_df = pd.DataFrame()
b_df = b_df.drop(columns=[col for col in b_df.columns if col.endswith('_drop')])
b_df[columns_to_float+assets] = (b_df[columns_to_float+assets]
                                    .replace(r"\.", "", regex=True)
                                    .replace(",", ".", regex=True)
                                    .astype(float))


In [4]:
fundaums_t = pd.DataFrame(index=dates, columns=assets)
fundaums_b = pd.DataFrame(index=dates, columns=assets)
topfunds_t = {a:{} for a in assets}
topfunds_b = {a:{} for a in assets}

In [5]:
for day in dates:
    tday = t_df[t_df['Tarih']==day]
    bday = b_df[b_df['Tarih']==day]
    for s in assets:
        tday.loc[:,s] *= tday.loc[:,'Fiyat'] * tday.loc[:,'Tedavüldeki Pay Sayısı']/100 
        fundaums_t.loc[day,s] = tday.loc[:,s].sum()
        topfunds_t[s][day]= tday.loc[:,['Fon Kodu', s]].sort_values(s, ascending=False).head(5).reset_index(drop=True)
        bday.loc[:,s] *= bday.loc[:,'Fiyat'] * bday.loc[:,'Tedavüldeki Pay Sayısı']/100
        fundaums_b.loc[day,s] = bday.loc[:,s].sum()
        topfunds_b[s][day]= bday.loc[:,['Fon Kodu', s]].sort_values(s, ascending=False).head(5).reset_index(drop=True)


/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tday.loc[:,s] *= tday.loc[:,'Fiyat'] * tday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bday.loc[:,s] *= bday.loc[:,'Fiyat'] * bday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWa

/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bday.loc[:,s] *= bday.loc[:,'Fiyat'] * bday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tday.loc[:,s] *= tday.loc[:,'Fiyat'] * tday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:8: SettingWithCopyWa

/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tday.loc[:,s] *= tday.loc[:,'Fiyat'] * tday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bday.loc[:,s] *= bday.loc[:,'Fiyat'] * bday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWa

/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bday.loc[:,s] *= bday.loc[:,'Fiyat'] * bday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tday.loc[:,s] *= tday.loc[:,'Fiyat'] * tday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:8: SettingWithCopyWa

/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tday.loc[:,s] *= tday.loc[:,'Fiyat'] * tday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bday.loc[:,s] *= bday.loc[:,'Fiyat'] * bday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWa

...

/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tday.loc[:,s] *= tday.loc[:,'Fiyat'] * tday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bday.loc[:,s] *= bday.loc[:,'Fiyat'] * bday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWa

/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tday.loc[:,s] *= tday.loc[:,'Fiyat'] * tday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bday.loc[:,s] *= bday.loc[:,'Fiyat'] * bday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWa

/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tday.loc[:,s] *= tday.loc[:,'Fiyat'] * tday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bday.loc[:,s] *= bday.loc[:,'Fiyat'] * bday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWa

/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tday.loc[:,s] *= tday.loc[:,'Fiyat'] * tday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bday.loc[:,s] *= bday.loc[:,'Fiyat'] * bday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWa

/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bday.loc[:,s] *= bday.loc[:,'Fiyat'] * bday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tday.loc[:,s] *= tday.loc[:,'Fiyat'] * tday.loc[:,'Tedavüldeki Pay Sayısı']/100
/var/folders/rf/gpy2j36s5qd53w_f572tfpzw0000gn/T/ipykernel_8101/3674681998.py:8: SettingWithCopyWa

In [6]:
fundaums = {s:fundaums_t.loc[:,s]+fundaums_b.loc[:,s] for s in assets}

In [16]:
fundaums

{'Hisse Senedi': 11.11.2024   616,464,902,989.93
 12.11.2024   621,422,759,787.27
 13.11.2024   616,295,780,387.06
 14.11.2024   623,025,304,896.01
 15.11.2024   627,623,048,191.40
                     ...        
 03.02.2025   721,438,545,897.56
 04.02.2025   710,146,022,087.49
 05.02.2025   714,131,501,370.50
 06.02.2025   707,836,062,104.38
 07.02.2025   708,738,597,281.19
 Name: Hisse Senedi, Length: 64, dtype: object,
 'Devlet Tahvili': 11.11.2024   162,171,588,967.83
 12.11.2024   162,962,622,476.01
 13.11.2024   164,716,485,965.30
 14.11.2024   164,738,222,787.37
 15.11.2024   166,121,640,035.55
                     ...        
 03.02.2025   425,531,585,618.40
 04.02.2025   429,082,497,873.76
 05.02.2025   429,740,214,036.87
 06.02.2025   431,938,013,306.35
 07.02.2025   430,026,697,227.55
 Name: Devlet Tahvili, Length: 64, dtype: object,
 'Hazine Bonosu': 11.11.2024       33,804,857.84
 12.11.2024       34,239,014.55
 13.11.2024       36,708,063.33
 14.11.2024       36,541,488.

In [18]:
fundaums[assets[1]].tail(30)

27.12.2024   242,114,651,174.67
30.12.2024   243,398,800,966.44
31.12.2024   243,848,386,346.13
02.01.2025   245,066,586,346.55
03.01.2025   245,486,153,422.62
06.01.2025   249,682,519,703.05
07.01.2025   253,638,844,657.95
08.01.2025   256,448,074,931.04
09.01.2025   259,546,609,076.98
10.01.2025   265,254,235,376.88
13.01.2025   271,047,353,318.10
14.01.2025   274,291,890,846.77
15.01.2025   312,927,938,800.15
16.01.2025   329,414,465,984.34
17.01.2025   335,884,598,480.40
20.01.2025   342,474,766,686.38
21.01.2025   346,694,983,663.87
22.01.2025   356,854,119,494.26
23.01.2025   368,408,765,372.08
24.01.2025   375,759,085,864.13
27.01.2025   323,325,883,138.98
28.01.2025   394,932,063,319.29
29.01.2025   401,030,323,642.70
30.01.2025   410,869,360,982.11
31.01.2025   418,189,337,982.52
03.02.2025   425,531,585,618.40
04.02.2025   429,082,497,873.76
05.02.2025   429,740,214,036.87
06.02.2025   431,938,013,306.35
07.02.2025   430,026,697,227.55
Name: Devlet Tahvili, dtype: object

In [8]:
def concat5(df1, df2, s):
    newdf = pd.concat([df1,df2])
    newdf = newdf.sort_values(s,ascending=False).head(5).reset_index(drop=True)
    return newdf

In [9]:
topfunds = {a:{} for a in assets}
for day in dates:
    for a in assets:
        topfunds[a][day] = concat5(topfunds_t[a][day],topfunds_b[a][day], a)

In [10]:
topfunds_b[assets[1]]['07.02.2025']

,Fon Kodu,Devlet Tahvili
0,VEI,"15,171,007,825.76"
1,AET,"15,141,627,864.53"
2,AEI,"13,942,994,696.92"
3,AMF,"10,021,945,391.86"
4,AZL,"9,958,598,558.59"


In [11]:
topfunds_t[assets[0]]['07.02.2025']

,Fon Kodu,Hisse Senedi
0,YAS,"10,814,711,135.11"
1,URV,"8,405,914,821.20"
2,IPE,"7,699,007,686.06"
3,TI2,"7,116,049,941.30"
4,ZJB,"7,112,686,924.38"


In [12]:
topfunds_b[assets[0]]['07.02.2025']

,Fon Kodu,Hisse Senedi
0,AZH,"29,372,305,112.91"
1,AH5,"20,067,786,853.28"
2,AEH,"19,879,256,780.70"
3,GEH,"14,071,051,139.93"
4,VEI,"10,276,358,451.93"


In [15]:
assets

['Hisse Senedi',
 'Devlet Tahvili',
 'Hazine Bonosu',
 'Döviz Kamu İç Borçlanma Araçları',
 'Finansman Bonosu',
 'Özel Sektör Tahvili',
 'Varlığa Dayalı Menkul Kıymetler',
 'Kamu Dış Borçlanma Araçları',
 'Özel Sektör Dış Borç. Araçları',
 'Takasbank Para Piyasası',
 'Kamu Kira Sertifikaları (TL)',
 'Kamu Kira Sertifikaları (Döviz)',
 'Özel Sektör Kira Sertifikaları',
 'Kamu Yurt Dışı Kira Sertifikaları',
 'Ö.S. Yurt Dışı Kira Sertifikaları',
 'Mevduat (TL)',
 'Mevduat (Döviz)',
 'Katılma Hesabı (TL)',
 'Katılma Hesabı (Döviz)',
 'Katılma Hesabı (Altın)',
 'Repo',
 'Ters-Repo',
 'Kıymetli Madenler',
 'Kıymetli Madenler Cinsinden BYF',
 'Kıymetli Madenler Kamu B. A.',
 'Kıymetli Madenler Kamu Kira Sertifika.',
 'Yabancı Kamu Borçlanma Araçları',
 'Yabancı Özel Sektör B.A.',
 'Yabancı Hisse Senedi',
 'Yabancı Borsa Yatırım Fonları',
 'Yatırım Fonları Katılma Payları',
 'BYF Katılma Payları',
 'Gayrimenkul Yatırım Fon Katılma Payları',
 'Girişim S. Yatırım Fon Katılma Payları',
 'Vadeli İ